# 00 -- Colab/Kaggle Setup & Drive Mount

Run this first in a fresh Colab/Kaggle session. It:
1. Clones the project from GitHub (if not already present in this session).
2. Mounts Google Drive (for large datasets: DFDC, FaceForensics++).
3. Installs `requirements-train.txt` (the PyTorch stack -- never needed
   for the served app itself, only for training/export).

**Dataset layout expected on Drive** (see `config/settings.yaml` ->
`data.root_drive` and `data.datasets_registry`):
```
/content/drive/MyDrive/ForensicAI_Lab/data/
    casia2/ comofod/ coverage/ imd2020/ ffpp_images/ dfdc_subset/ ffpp_videos/
    images/ masks/ labels.csv
```
If your datasets are sitting directly under `MyDrive/ForensicAI_Lab/` (no
`data/` subfolder) and/or still zipped, see
`00b_prepare_local_datasets.ipynb`'s "Option C" cell, which extracts them
into the right place automatically.

In [ ]:
import sys, os

def _is_colab():
    try:
        import google.colab  # noqa
        return True
    except ImportError:
        return False

IS_COLAB = _is_colab()
IS_KAGGLE = (not IS_COLAB) and (os.path.exists("/kaggle/input") or "KAGGLE_KERNEL_RUN_TYPE" in os.environ)
PROJECT_ROOT = "/kaggle/working/ForensicAI_Lab" if IS_KAGGLE else "/content/ForensicAI_Lab"

if not os.path.exists(PROJECT_ROOT):
    !git clone https://github.com/MRoy03/Forensic_LAB.git {PROJECT_ROOT}

sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

from notebooks.utils.colab_common import mount_drive, install_requirements, is_colab

if is_colab():
    mount_drive()
    install_requirements(PROJECT_ROOT)
elif IS_KAGGLE:
    !pip install -q -r requirements-train.txt
else:
    print("Not in Colab/Kaggle -- assuming requirements-train.txt is already installed locally.")

In [ ]:
from config.loader import get_settings, resolve_dataset_root

settings = get_settings()
dataset_root = resolve_dataset_root(settings)
print("Using dataset root:", dataset_root)

for entry in settings.data.datasets_registry:
    subdir = dataset_root / entry.local_subdir
    status = "found" if subdir.exists() else "MISSING (download/place it here)"
    print(f"  {entry.name:25s} ({entry.type:5s}) -> {subdir}  [{status}]")